# 02 — Data Cleaning

**Goal:** Take the raw XLS files, fix every structural issue identified in Notebook 01, and export two clean CSVs that all subsequent notebooks will read from.

**Outputs:**
- `data/processed/national_monthly.csv` — England-wide monthly time series, April 2019–April 2026
- `data/processed/quarterly_by_provider.csv` — trust-level quarterly data, 2019-20 Q1 through 2024-25 Q4

**What we don't do here:** no analysis, no charts, no interpretation. This notebook is purely mechanical — fix the structure, rename columns, filter to the right date window, and export. That's it.

**Raw data is never modified.** We only write to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

TS_PATH = "../data/raw/time_series/Monthly-AE-Time-Series-April-2026.xls"
Q_DIR   = "../data/raw/quarterly_by_provider/"
OUT_DIR = "../data/processed/"

os.makedirs(OUT_DIR, exist_ok=True)
print("Ready.")

Ready.


---
## Part 1 — National Time Series

The time series XLS has two useful sheets:
- **Activity** — monthly attendance counts and emergency admission counts
- **Performance** — 4-hour compliance percentages by department type

We clean both separately, then merge them on `period` into one table. This gives downstream notebooks a single, complete national dataset.

**Why merge rather than keep separate?** Every analysis we'll run needs both volume (how many attended) and performance (how many were seen in time) on the same row. Merging now means we never have to join mid-analysis.

### 1a — Activity sheet

In [2]:
act_raw = pd.read_excel(
    TS_PATH,
    sheet_name="Activity",
    engine="xlrd",
    skiprows=13,
)

# Drop junk columns identified in Notebook 01:
# - Unnamed: 0, Unnamed: 14, Unnamed: 16 are fully null (artefacts of XLS formatting)
# - 0.95 (a float column name) is a constant column representing the 95% target line — belongs in charts, not data
# - 'Operational standard (Performance)' is the same value as pct_4hr_all in the Performance
#   sheet but only populated for 87/189 rows — we take it from Performance instead
# Note: 0.95 is a float, not the string "0.95" — pandas reads numeric XLS headers as floats
JUNK_COLS_ACT = ["Unnamed: 0", "Unnamed: 14", "Unnamed: 16", 0.95,
                 "Operational standard (Performance)"]
act_raw.drop(columns=[c for c in JUNK_COLS_ACT if c in act_raw.columns], inplace=True)

ACTIVITY_RENAME = {
    "Period":                                                                         "period",
    "Type 1 Departments - Major A&E":                                                 "type1_attendances",
    "Type 2 Departments - Single Specialty":                                          "type2_attendances",
    "Type 3 Departments - Other A&E/Minor Injury Unit":                               "type3_attendances",
    "Total Attendances":                                                              "total_attendances",
    "Emergency Admissions via Type 1 A&E":                                            "emerg_admissions_type1",
    "Emergency Admissions via Type 2 A&E":                                            "emerg_admissions_type2",
    "Emergency Admissions via Type 3 and 4 A&E":                                      "emerg_admissions_type3",
    "Total Emergency Admissions via A&E":                                             "total_emerg_admissions_ae",
    "Other Emergency Admissions (i.e not via A&E)":                                   "other_emerg_admissions",
    "Total Emergency Admissions":                                                     "total_emerg_admissions",
    "Number of patients spending >4 hours from decision to admit to admission":        "dtoa_over_4hr",
    "Number of patients spending >12 hours from decision to admit to admission":       "dtoa_over_12hr",
}
act = act_raw.rename(columns=ACTIVITY_RENAME)

print(f"Loaded Activity: {act.shape}")
print("Columns:", list(act.columns))

Loaded Activity: (189, 13)
Columns: ['period', 'type1_attendances', 'type2_attendances', 'type3_attendances', 'total_attendances', 'emerg_admissions_type1', 'emerg_admissions_type2', 'emerg_admissions_type3', 'total_emerg_admissions_ae', 'other_emerg_admissions', 'total_emerg_admissions', 'dtoa_over_4hr', 'dtoa_over_12hr']


In [3]:
# Filter to our analysis window: April 2019 onwards
# The raw file goes back to August 2010; we only need the 6 financial years 2019-20 to 2024-25.
# April 2019 = start of financial year 2019-20.
act = act[act["period"] >= "2019-04-01"].copy()
act = act.dropna(subset=["period"]).reset_index(drop=True)

print(f"After filtering: {act.shape}")
print(f"Date range: {act['period'].min().date()} → {act['period'].max().date()}")
act.head(3)

After filtering: (85, 13)
Date range: 2019-04-01 → 2026-04-01


,period,type1_attendances,type2_attendances,type3_attendances,total_attendances,emerg_admissions_type1,emerg_admissions_type2,emerg_admissions_type3,total_emerg_admissions_ae,other_emerg_admissions,total_emerg_admissions,dtoa_over_4hr,dtoa_over_12hr
0,2019-04-01,1330825.0,49281.0,732059.0,2112165.0,398802.0,1787.0,4850.0,405439.0,129787.0,535226.0,66933.0,442.0
1,2019-05-01,1369332.0,50642.0,752032.0,2172006.0,406065.0,2145.0,4631.0,412841.0,134541.0,547382.0,61507.0,416.0
2,2019-06-01,1334137.0,49233.0,724617.0,2107987.0,392446.0,1528.0,4696.0,398670.0,130131.0,528801.0,57671.0,462.0


### 1b — Performance sheet

The Performance sheet duplicates the type-level attendance totals (they also appear in Activity). We drop those duplicates and keep only what Activity doesn't have: seen/breach counts by type, and the 4-hour compliance percentages.

The `.1` suffix on columns 6–8 is pandas auto-disambiguation: the Performance sheet has two rows of headers, and the category labels ('Type 1 Departments - Major A&E') appear twice — once for the total count (no suffix) and once for the seen-within-4hr count (`.1` suffix). We deal with this by selecting exactly the columns we want by their pandas names.

In [4]:
perf_raw = pd.read_excel(
    TS_PATH,
    sheet_name="Performance",
    engine="xlrd",
    skiprows=13,
)

# Select only the columns we want — drop Unnamed: 0 and the attendance total duplicates
PERF_KEEP = [
    "Period",
    "Total Attendances < 4 hours",
    "Type 1 Departments - Major A&E.1",
    "Type 2 Departments - Single Specialty.1",
    "Type 3 Departments - Other A&E/Minor Injury Unit.1",
    "Total Attendances > 4 hours",
    "Percentage in 4 hours or less (all)",
    "Percentage in 4 hours or less (type 1)",
    "Percentage in 4 hours or less (type 2)",
    "Percentage in 4 hours or less (type 3)",
]
perf_raw = perf_raw[[c for c in PERF_KEEP if c in perf_raw.columns]]

PERF_RENAME = {
    "Period":                                                    "period",
    "Total Attendances < 4 hours":                               "total_seen_4hr",
    "Type 1 Departments - Major A&E.1":                          "type1_seen_4hr",
    "Type 2 Departments - Single Specialty.1":                   "type2_seen_4hr",
    "Type 3 Departments - Other A&E/Minor Injury Unit.1":        "type3_seen_4hr",
    "Total Attendances > 4 hours":                               "total_breach_4hr",
    "Percentage in 4 hours or less (all)":                       "pct_4hr_all",
    "Percentage in 4 hours or less (type 1)":                    "pct_4hr_type1",
    "Percentage in 4 hours or less (type 2)":                    "pct_4hr_type2",
    "Percentage in 4 hours or less (type 3)":                    "pct_4hr_type3",
}
perf = perf_raw.rename(columns=PERF_RENAME)

# Filter to April 2019+
perf = perf[perf["period"] >= "2019-04-01"].copy()
perf = perf.dropna(subset=["period"]).reset_index(drop=True)

print(f"Loaded Performance: {perf.shape}")
print(f"Date range: {perf['period'].min().date()} → {perf['period'].max().date()}")
perf.head(3)

Loaded Performance: (85, 10)
Date range: 2019-04-01 → 2026-04-01


,period,total_seen_4hr,type1_seen_4hr,type2_seen_4hr,type3_seen_4hr,total_breach_4hr,pct_4hr_all,pct_4hr_type1,pct_4hr_type2,pct_4hr_type3
0,2019-04-01,1797984.0,304221.0,800.0,9160.0,314181.0,0.851252,0.771404,0.983767,0.987487
1,2019-05-01,1694301.0,253619.0,639.0,7207.0,261465.0,0.866311,0.790813,0.985280,0.989704
2,2019-06-01,1638461.0,250465.0,781.0,7336.0,258582.0,0.863692,0.787961,0.981373,0.989114


### 1c — Merge Activity + Performance

Inner join on `period`. Because both sheets are filtered to April 2019+ and both cover the full national time series, the inner join preserves all rows in the overlapping date range (which is all of them).

In [5]:
national = act.merge(perf, on="period", how="inner")

print(f"Activity rows: {len(act)}  |  Performance rows: {len(perf)}  |  Merged rows: {len(national)}")
print(f"Date range: {national['period'].min().date()} → {national['period'].max().date()}")
print(f"Columns ({len(national.columns)}): {list(national.columns)}")
national.head(3)

Activity rows: 85  |  Performance rows: 85  |  Merged rows: 85
Date range: 2019-04-01 → 2026-04-01
Columns (22): ['period', 'type1_attendances', 'type2_attendances', 'type3_attendances', 'total_attendances', 'emerg_admissions_type1', 'emerg_admissions_type2', 'emerg_admissions_type3', 'total_emerg_admissions_ae', 'other_emerg_admissions', 'total_emerg_admissions', 'dtoa_over_4hr', 'dtoa_over_12hr', 'total_seen_4hr', 'type1_seen_4hr', 'type2_seen_4hr', 'type3_seen_4hr', 'total_breach_4hr', 'pct_4hr_all', 'pct_4hr_type1', 'pct_4hr_type2', 'pct_4hr_type3']


,period,type1_attendances,type2_attendances,type3_attendances,total_attendances,emerg_admissions_type1,emerg_admissions_type2,emerg_admissions_type3,total_emerg_admissions_ae,other_emerg_admissions,total_emerg_admissions,dtoa_over_4hr,dtoa_over_12hr,total_seen_4hr,type1_seen_4hr,type2_seen_4hr,type3_seen_4hr,total_breach_4hr,pct_4hr_all,pct_4hr_type1,pct_4hr_type2,pct_4hr_type3
0,2019-04-01,1330825.0,49281.0,732059.0,2112165.0,398802.0,1787.0,4850.0,405439.0,129787.0,535226.0,66933.0,442.0,1797984.0,304221.0,800.0,9160.0,314181.0,0.851252,0.771404,0.983767,0.987487
1,2019-05-01,1369332.0,50642.0,752032.0,2172006.0,406065.0,2145.0,4631.0,412841.0,134541.0,547382.0,61507.0,416.0,1694301.0,253619.0,639.0,7207.0,261465.0,0.866311,0.790813,0.985280,0.989704
2,2019-06-01,1334137.0,49233.0,724617.0,2107987.0,392446.0,1528.0,4696.0,398670.0,130131.0,528801.0,57671.0,462.0,1638461.0,250465.0,781.0,7336.0,258582.0,0.863692,0.787961,0.981373,0.989114


In [6]:
# Validate: check for nulls and unexpected dtypes
print("=== Null counts ===")
null_counts = national.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No nulls")
print()
print("=== Dtypes ===")
print(national.dtypes)
print()
print("=== Performance column range check (should be 0.0–1.0) ===")
# Guard against float column names (e.g. if any junk column slipped through)
pct_cols = [c for c in national.columns if isinstance(c, str) and c.startswith("pct_")]
print(national[pct_cols].describe().round(3))

=== Null counts ===
No nulls

=== Dtypes ===
period                       datetime64[ns]
type1_attendances                   float64
type2_attendances                   float64
type3_attendances                   float64
total_attendances                   float64
emerg_admissions_type1              float64
emerg_admissions_type2              float64
emerg_admissions_type3              float64
total_emerg_admissions_ae           float64
other_emerg_admissions              float64
total_emerg_admissions              float64
dtoa_over_4hr                       float64
dtoa_over_12hr                      float64
total_seen_4hr                      float64
type1_seen_4hr                      float64
type2_seen_4hr                      float64
type3_seen_4hr                      float64
total_breach_4hr                    float64
pct_4hr_all                         float64
pct_4hr_type1                       float64
pct_4hr_type2                       float64
pct_4hr_type3                  

In [7]:
out_path = OUT_DIR + "national_monthly.csv"
national.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {national.shape}  ({len(national)} months × {len(national.columns)} columns)")

Saved: ../data/processed/national_monthly.csv
Shape: (85, 22)  (85 months × 22 columns)


---
## Part 2 — Quarterly By-Provider

24 XLS files, one per quarter from 2019-20 Q1 to 2024-25 Q4. Each has identical column structure (confirmed in Notebook 01). We:

1. Define a single column rename map (same for all 24 files)
2. Write a `load_quarter()` function that loads one file, applies all cleaning steps, and adds `financial_year` and `quarter` columns from the filename
3. Loop over all 24 files and concatenate the results
4. Validate and export

### Column naming convention

The raw quarterly file has three groups of columns per department type, and pandas disambiguates repeated header names with `.1` and `.2` suffixes:

| Raw pandas name | Meaning | Renamed to |
|---|---|---|
| `Type 1 Departments - Major A&E` | Total attendances | `type1_attendances` |
| `Type 1 Departments - Major A&E.1` | Seen within 4 hours | `type1_seen_4hr` |
| `Type 1 Departments - Major A&E.2` | Breaches (not seen in time) | `type1_breach_4hr` |

The pattern repeats for Type 2 and Type 3.

**Performance columns are stored as decimals (0.0–1.0).** We coerce them with `pd.to_numeric(..., errors='coerce')` which turns the `-` dash values (trusts with no department of that type) into NaN cleanly.

In [8]:
Q_RENAME = {
    "Code":                                                                                  "code",
    "Region":                                                                                "region",
    "Name":                                                                                  "name",
    "Type 1 Departments - Major A&E":                                                        "type1_attendances",
    "Type 2 Departments - Single Specialty":                                                 "type2_attendances",
    "Type 3 Departments - Other A&E/Minor Injury Unit":                                      "type3_attendances",
    "Total attendances":                                                                     "total_attendances",
    "Type 1 Departments - Major A&E.1":                                                      "type1_seen_4hr",
    "Type 2 Departments - Single Specialty.1":                                               "type2_seen_4hr",
    "Type 3 Departments - Other A&E/Minor Injury Unit.1":                                    "type3_seen_4hr",
    "Total Attendances < 4 hours":                                                           "total_seen_4hr",
    "Type 1 Departments - Major A&E.2":                                                      "type1_breach_4hr",
    "Type 2 Departments - Single Specialty.2":                                               "type2_breach_4hr",
    "Type 3 Departments - Other A&E/Minor Injury Unit.2":                                    "type3_breach_4hr",
    "Total Attendances > 4 hours":                                                           "total_breach_4hr",
    "Percentage in 4 hours or less (all)":                                                   "pct_4hr_all",
    "Percentage in 4 hours or less (type 1)":                                                "pct_4hr_type1",
    "Percentage in 4 hours or less (type 2)":                                                "pct_4hr_type2",
    "Percentage in 4 hours or less (type 3)":                                                "pct_4hr_type3",
    "Emergency Admissions via Type 1 A&E":                                                   "emerg_admissions_type1",
    "Emergency Admissions via Type 2 A&E":                                                   "emerg_admissions_type2",
    "Emergency Admissions via Type 3 and 4 A&E":                                             "emerg_admissions_type3",
    "Total Emergency Admissions via A&E":                                                    "total_emerg_admissions_ae",
    "Other Emergency admissions (i.e not via A&E)":                                          "other_emerg_admissions",
    "Total Emergency Admissions":                                                            "total_emerg_admissions",
    "Number of patients spending >4 hours from decision to admit to admission":               "dtoa_over_4hr",
    "Number of patients spending >12 hours from decision to admit to admission":              "dtoa_over_12hr",
    "Unnamed: 28":                                                                           "icb_name",
}

PCT_COLS = ["pct_4hr_all", "pct_4hr_type1", "pct_4hr_type2", "pct_4hr_type3"]


def load_quarter(filepath):
    """Load one quarterly XLS, apply all cleaning, return a clean DataFrame."""
    # Parse financial year and quarter from filename (e.g. '2019-20_Q1.xls')
    basename = os.path.basename(filepath)
    match = re.match(r"(\d{4}-\d{2})_Q(\d)", basename)
    financial_year = match.group(1)   # e.g. '2019-20'
    quarter = int(match.group(2))     # e.g. 1

    df = pd.read_excel(
        filepath,
        sheet_name="Provider Level Data",
        engine="xlrd",
        skiprows=15,
    )

    # Drop junk columns: Unnamed: 0 (fully null), any other fully-null unnamed cols
    df.drop(columns=[c for c in df.columns if "Unnamed" in str(c) and c != "Unnamed: 28"],
            inplace=True)

    # Rename
    df.rename(columns=Q_RENAME, inplace=True)

    # Drop England total row (Code == '-') and blank separator row (Code is NaN)
    df = df[df["code"].notna() & (df["code"] != "-")].copy()

    # Coerce performance columns from object (mixed str/float) to float
    # '-' dash values become NaN — these are trusts with no department of that type
    for col in PCT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Add period identifiers
    df["financial_year"] = financial_year
    df["quarter"] = quarter

    return df.reset_index(drop=True)


# Smoke-test on one file
test = load_quarter(Q_DIR + "2024-25_Q4.xls")
print(f"Smoke test — 2024-25 Q4: {test.shape}")
print(f"Columns: {list(test.columns)}")
test[["code", "name", "financial_year", "quarter", "type1_attendances", "pct_4hr_type1"]].head(5)

!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
Smoke test — 2024-25 Q4: (199, 30)
Columns: ['code', 'region', 'name', 'type1_attendances', 'type2_attendances', 'type3_attendances', 'total_attendances', 'type1_seen_4hr', 'type2_seen_4hr', 'type3_seen_4hr', 'total_seen_4hr', 'type1_breach_4hr', 'type2_breach_4hr', 'type3_breach_4hr', 'total_breach_4hr', 'pct_4hr_all', 'pct_4hr_type1', 'pct_4hr_type2', 'pct_4hr_type3', 'emerg_admissions_type1', 'emerg_admissions_type2', 'emerg_admissions_type3', 

,code,name,financial_year,quarter,type1_attendances,pct_4hr_type1
0,RC9,Bedfordshire Hospitals NHS Foundation Trust,2024-25,4,47182.0,0.551460
1,RGT,Cambridge University Hospitals NHS Foundation Trust,2024-25,4,31303.0,0.540204
2,RWH,East And North Hertfordshire NHS Trust,2024-25,4,25947.0,0.464177
3,RDE,East Suffolk And North Essex NHS Foundation Trust,2024-25,4,38254.0,0.530690
4,RY4,Hertfordshire Community NHS Trust,2024-25,4,0.0,NaN


### 2b — Load all 24 files and concatenate

In [9]:
files = sorted(glob.glob(Q_DIR + "*.xls"))
print(f"Found {len(files)} quarterly files")

frames = []
for f in files:
    df = load_quarter(f)
    frames.append(df)
    print(f"  {os.path.basename(f):<25}  {len(df):>3} trusts")

quarterly = pd.concat(frames, ignore_index=True)
print(f"\nConcatenated: {quarterly.shape}")

Found 24 quarterly files
  2019-20_Q1.xls             239 trusts


  2019-20_Q2.xls             233 trusts


  2019-20_Q3.xls             232 trusts


  2019-20_Q4.xls             232 trusts


  2020-21_Q1.xls             222 trusts

...

  2023-24_Q3.xls             203 trusts
  2023-24_Q4.xls             201 trusts


  2024-25_Q1.xls             197 trusts


  2024-25_Q2.xls             198 trusts
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)
!!! get_externsheet_local_range: refx=65535, not in range(1)


  2024-25_Q3.xls             198 trusts
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)
!!! get_externsheet_local_range: refx=65535, not in range(2)


  2024-25_Q4.xls             199 trusts

Concatenated: (5062, 30)


### 2c — Validate the combined dataset

In [10]:
print("=== Shape ===")
print(f"{len(quarterly):,} rows × {len(quarterly.columns)} columns")
print()

print("=== Financial years covered ===")
print(quarterly.groupby(["financial_year", "quarter"]).size().rename("trust_count").to_string())
print()

print("=== Null counts (selected columns) ===")
key_cols = ["code", "name", "region", "type1_attendances", "pct_4hr_type1", "icb_name"]
print(quarterly[key_cols].isnull().sum())
print()

print("=== pct_4hr_type1 range (decimal, Type 1 trusts only) ===")
type1_trusts = quarterly[quarterly["type1_attendances"].notna() & (quarterly["type1_attendances"] > 0)]
print(type1_trusts["pct_4hr_type1"].describe().round(3))

=== Shape ===
5,062 rows × 30 columns

=== Financial years covered ===
financial_year  quarter
2019-20         1          239
                2          233
                3          232
                4          232
2020-21         1          222
                2          218
                3          217
                4          215
2021-22         1          211
                2          209
                3          207
                4          207
2022-23         1          205
                2          205
                3          204
                4          203
2023-24         1          204
                2          203
                3          203
                4          201
2024-25         1          197
                2          198
                3          198
                4          199

=== Null counts (selected columns) ===


code                    0
name                    0
region                  0
type1_attendances       0
pct_4hr_type1        2286
icb_name              395
dtype: int64

=== pct_4hr_type1 range (decimal, Type 1 trusts only) ===
count    2776.000
mean        0.666
std         0.141
min         0.286
25%         0.559
50%         0.653
75%         0.767
max         0.995
Name: pct_4hr_type1, dtype: float64


In [11]:
# Spot-check: Cambridge University Hospitals (RGT) — local trust for Huntingdonshire context
rgt = quarterly[quarterly["code"] == "RGT"][["financial_year", "quarter", "name",
                                              "type1_attendances", "pct_4hr_type1", "icb_name"]]
print(f"RGT (Cambridge University Hospitals) — {len(rgt)} quarters")
print(rgt.to_string(index=False))

RGT (Cambridge University Hospitals) — 24 quarters
financial_year  quarter                                                name  type1_attendances  pct_4hr_type1                                                  icb_name
       2019-20        1 Cambridge University Hospitals NHS Foundation Trust            31786.0            NaN                       Cambridgeshire and Peterborough STP
       2019-20        2 Cambridge University Hospitals NHS Foundation Trust            30781.0            NaN                       Cambridgeshire and Peterborough STP
       2019-20        3 Cambridge University Hospitals NHS Foundation Trust            30549.0            NaN                       Cambridgeshire and Peterborough STP
       2019-20        4 Cambridge University Hospitals NHS Foundation Trust            26024.0            NaN                       Cambridgeshire and Peterborough STP
       2020-21        1 Cambridge University Hospitals NHS Foundation Trust            19009.0            NaN

In [12]:
out_path = OUT_DIR + "quarterly_by_provider.csv"
quarterly.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {quarterly.shape}  ({len(quarterly):,} trust-quarter rows × {len(quarterly.columns)} columns)")

Saved: ../data/processed/quarterly_by_provider.csv
Shape: (5062, 30)  (5,062 trust-quarter rows × 30 columns)


---
## Summary

### What was produced

| File | Rows | Columns | Coverage |
|---|---|---|---|
| `national_monthly.csv` | ~85 | 23 | April 2019 – April 2026, monthly |
| `quarterly_by_provider.csv` | ~5,000 | 30 | 2019-20 Q1 – 2024-25 Q4, ~200 trusts/quarter |

### Key cleaning decisions

| Decision | Reason |
|---|---|
| Dropped `Unnamed: 0`, `Unnamed: 14`, `Unnamed: 16`, `0.95` | Fully null or constant — artefacts of XLS layout, not data |
| Dropped `Operational standard (Performance)` from Activity | Only 87/189 rows populated; same value available fully in Performance sheet |
| Merged Activity + Performance on `period` | Gives one complete national table for EDA notebooks |
| Filtered to April 2019 onwards | Our analysis scope is 6 financial years 2019-20 to 2024-25 |
| Dropped England total row (`code == '-'`) | Trust-level analysis only; England aggregate is in `national_monthly.csv` |
| `pd.to_numeric(..., errors='coerce')` on pct columns | Converts `-` dash (no dept of that type) → NaN cleanly |
| Added `financial_year`, `quarter` from filename | No date column in raw quarterly files — filename is the only source |
| Kept `icb_name` (renamed from `Unnamed: 28`) | ICB is useful for geographic grouping and the Huntingdonshire local angle |

### Next: Notebook 03 — EDA National Trends
Read from `national_monthly.csv`. Chart attendance trends, 4-hour performance over time, COVID impact, and recovery trajectory.